In [3]:
import os
import glob
import subprocess
import pandas as pd
import numpy as np
from Bio import SeqIO

WORK_DIR = os.path.expanduser("~/LB2_Project")
os.chdir(WORK_DIR)

RANDOM_SEED = 42
os.makedirs("cluster_output", exist_ok=True)
os.makedirs("mmseqs_tmp", exist_ok=True)

def find_mmseqs_rep_fasta(prefix):
    expected = f"{prefix}_rep_seq.fasta"
    if os.path.exists(expected):
        return expected
    parent = os.path.dirname(prefix) or "."
    stem = os.path.basename(prefix)
    candidates = sorted(p for p in glob.glob(os.path.join(parent, f"{stem}*")) if p.endswith(".fasta"))
    preferred = [p for p in candidates if "rep" in os.path.basename(p).lower()]
    if preferred:
        return preferred[0]
    if candidates:
        return candidates[0]
    raise FileNotFoundError(f"No FASTA output found for MMseqs prefix '{prefix}'.")

n_pos_fasta = sum(1 for _ in SeqIO.parse("positive.fasta", "fasta"))
n_neg_fasta = sum(1 for _ in SeqIO.parse("negative.fasta", "fasta"))
df_pos_raw = pd.read_csv("positive.tsv", sep="\t")
df_neg_raw = pd.read_csv("negative.tsv", sep="\t")

assert n_pos_fasta == len(df_pos_raw), "Mismatch in positive FASTA vs TSV"
assert n_neg_fasta == len(df_neg_raw), "Mismatch in negative FASTA vs TSV"
assert len(set(df_pos_raw["Accession"]) & set(df_neg_raw["Accession"])) == 0, "Overlap in raw sets!"

print("Running MMseqs2 clustering on positive.fasta...")
subprocess.run([
    "mmseqs", "easy-cluster", "positive.fasta", "cluster_output/pos", "mmseqs_tmp",
    "--min-seq-id", "0.3", "-c", "0.4", "--cov-mode", "0", "--cluster-mode", "1", "-v", "1"
], check=True)

print("Running MMseqs2 clustering on negative.fasta...")
subprocess.run([
    "mmseqs", "easy-cluster", "negative.fasta", "cluster_output/neg", "mmseqs_tmp",
    "--min-seq-id", "0.3", "-c", "0.4", "--cov-mode", "0", "--cluster-mode", "1", "-v", "1"
], check=True)

pos_reps = list(SeqIO.parse(find_mmseqs_rep_fasta("cluster_output/pos"), "fasta"))
neg_reps = list(SeqIO.parse(find_mmseqs_rep_fasta("cluster_output/neg"), "fasta"))

pos_rep_ids = set(rec.id for rec in pos_reps)
neg_rep_ids = set(rec.id for rec in neg_reps)
assert len(pos_rep_ids & neg_rep_ids) == 0, "Overlap in representative IDs!"

df_pos = df_pos_raw[df_pos_raw["Accession"].isin(pos_rep_ids)].copy().reset_index(drop=True)
df_neg = df_neg_raw[df_neg_raw["Accession"].isin(neg_rep_ids)].copy().reset_index(drop=True)

df_pos.to_csv("filtered_positive.tsv", sep="\t", index=False)
df_neg.to_csv("filtered_negative.tsv", sep="\t", index=False)

df_pos["label"] = 1
df_neg["label"] = 0
df_all = pd.concat([df_pos, df_neg], ignore_index=True)

def stratified_split(df, train_frac=0.8, seed=42):
    rng = np.random.default_rng(seed)
    train_indices, bench_indices = [], []
    for label_val in sorted(df["label"].unique()):
        idx = df[df["label"] == label_val].index.to_numpy()
        rng.shuffle(idx)
        cut = int(len(idx) * train_frac)
        train_indices.extend(idx[:cut])
        bench_indices.extend(idx[cut:])
    return df.loc[train_indices].reset_index(drop=True), df.loc[bench_indices].reset_index(drop=True)

def assign_cv_folds(df, n_folds=5, seed=42):
    rng = np.random.default_rng(seed)
    df = df.copy()
    df["fold"] = -1
    for label_val in sorted(df["label"].unique()):
        idx = df[df["label"] == label_val].index.to_numpy()
        rng.shuffle(idx)
        for fold_num, chunk in enumerate(np.array_split(idx, n_folds)):
            df.loc[chunk, "fold"] = fold_num
    assert (df["fold"] == -1).sum() == 0, "Unassigned rows!"
    return df

df_train, df_bench = stratified_split(df_all, train_frac=0.8, seed=RANDOM_SEED)
df_train = assign_cv_folds(df_train, n_folds=5, seed=RANDOM_SEED)

df_train.to_csv("training_with_folds.tsv", sep="\t", index=False)
df_bench.to_csv("benchmarking_set.tsv", sep="\t", index=False)

print("\n==========================================================")
print("  DATA PREPARATION SUMMARY")
print("==========================================================")
print(f"  Raw sequences          : Pos={n_pos_fasta:,} | Neg={n_neg_fasta:,}")
print(f"  After MMseqs2 cluster  : Pos={len(df_pos):,} | Neg={len(df_neg):,} | Total={len(df_all):,}")
print(f"  Training set (80%)     : Pos={(df_train['label']==1).sum():,} | Neg={(df_train['label']==0).sum():,} | Total={len(df_train):,}")
print(f"  Benchmarking set (20%) : Pos={(df_bench['label']==1).sum():,} | Neg={(df_bench['label']==0).sum():,} | Total={len(df_bench):,}")
print("==========================================================")

Running MMseqs2 clustering on positive.fasta...
Running MMseqs2 clustering on negative.fasta...

  DATA PREPARATION SUMMARY
  Raw sequences          : Pos=2,953 | Neg=20,974
  After MMseqs2 cluster  : Pos=1,100 | Neg=9,082 | Total=10,182
  Training set (80%)     : Pos=880 | Neg=7,265 | Total=8,145
  Benchmarking set (20%) : Pos=220 | Neg=1,817 | Total=2,037
